In [ ]:
# ============================================================
# PHASE 4 END-TO-END MODIFIED
# MODEL: TimeAwareAttentionLSTM + ReLU-GELU
# Fixes:
# 1. No local copy from Drive
# 2. Robust real batch discovery
# 3. Lightweight validation only
# 4. Classification + Top-K metrics
# ============================================================

# ============================================================
# 0. MOUNT DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import gc
import json
import time
import pickle
import random
import warnings
import re
from glob import glob

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 2. REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if not torch.cuda.is_available():
    print("WARNING: CUDA is not available. Training on CPU will be very slow.")

# ============================================================
# 3. CONFIG
# ============================================================

PHASE3_DIR = "/content/drive/MyDrive/Instacart/phase3_outputs_final"

PHASE4_DIR = "/content/drive/MyDrive/Instacart/phase4_outputs_lstm_relu_gelu_stable"
os.makedirs(PHASE4_DIR, exist_ok=True)

NUM_EPOCHS = 5
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5
DROPOUT = 0.2
VAL_SPLIT = 0.10

INNER_BATCH_SIZE = 256
TOPK_INNER_BATCH_SIZE = 64
PRINT_EVERY_FILES = 25

EMBED_DIM_PRODUCT = 64
EMBED_DIM_AISLE = 8
EMBED_DIM_DEPT = 4
EMBED_DIM_DOW = 4
EMBED_DIM_HOUR = 4
TIME_FEATURE_DIM = 4
RNN_HIDDEN_DIM = 64
DENSE_DIM = 64
RECENCY_BETA_INIT = 0.10

MODEL_VARIANTS = [
    {
        "name": "Light_TimeAwareAttentionLSTM_ReLU_GELU",
        "rnn_type": "LSTM",
        "activation": "relu_gelu"
    }
]

# Keep False unless you intentionally increase NUM_EPOCHS and want continuation
FORCE_CONTINUE_TRAINING = False

print("=" * 80)
print("Using Phase 3 directly from Drive:")
print(PHASE3_DIR)
print("=" * 80)

# ============================================================
# 4. HELPERS
# ============================================================

def print_line():
    print("=" * 80)

def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)

def load_pickle(path, retries=3, sleep_sec=3):
    """
    Retry wrapper to reduce failures from temporary Google Drive read issues.
    If Drive is truly disconnected, it will still fail after retries.
    """
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception as e:
            last_err = e
            print(f"Read failed attempt {attempt}/{retries}: {path}")
            print("Error:", e)
            time.sleep(sleep_sec)

    raise last_err

def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ============================================================
# 5. LOAD METADATA
# ============================================================

metadata_candidates = glob(os.path.join(PHASE3_DIR, "phase3_metadata*.pkl"))

if len(metadata_candidates) == 0:
    raise FileNotFoundError(f"No Phase 3 metadata found inside {PHASE3_DIR}")

stable_metadata = os.path.join(PHASE3_DIR, "phase3_metadata.pkl")

if os.path.exists(stable_metadata):
    metadata_path = stable_metadata
else:
    metadata_path = max(metadata_candidates, key=os.path.getmtime)

metadata = load_pickle(metadata_path)

print_line()
print("Loaded metadata from:", metadata_path)

for k, v in metadata.items():
    if isinstance(v, (int, float, str)):
        print(f"{k}: {v}")

RUN_ID = metadata["RUN_ID"]

print_line()
print("Metadata RUN_ID:", RUN_ID)

# ============================================================
# 6. ROBUST REAL BATCH FILE DISCOVERY
# Avoids train_batch_index / test_batch_index issue
# ============================================================

print_line()
print("Discovering real Phase 3 batch files...")

all_pkl_files = sorted(glob(os.path.join(PHASE3_DIR, "*.pkl")))

train_pattern_current = re.compile(rf"^train_batch_\d{{4}}_{RUN_ID}\.pkl$")
test_pattern_current  = re.compile(rf"^test_batch_\d{{4}}_{RUN_ID}\.pkl$")

raw_train_batch_files = [
    f for f in all_pkl_files
    if train_pattern_current.match(os.path.basename(f))
]

raw_test_batch_files = [
    f for f in all_pkl_files
    if test_pattern_current.match(os.path.basename(f))
]

print("Real train batches for metadata RUN_ID:", len(raw_train_batch_files))
print("Real test batches for metadata RUN_ID :", len(raw_test_batch_files))

if len(raw_train_batch_files) == 0 or len(raw_test_batch_files) == 0:
    print("\nNo real batch files found for metadata RUN_ID.")
    print("Searching all real batch files from any RUN_ID...")

    train_pattern_any = re.compile(r"^train_batch_\d{4}_\d{8}_\d{6}\.pkl$")
    test_pattern_any  = re.compile(r"^test_batch_\d{4}_\d{8}_\d{6}\.pkl$")

    raw_train_batch_files = [
        f for f in all_pkl_files
        if train_pattern_any.match(os.path.basename(f))
    ]

    raw_test_batch_files = [
        f for f in all_pkl_files
        if test_pattern_any.match(os.path.basename(f))
    ]

if len(raw_train_batch_files) > 0:
    sample_name = os.path.basename(raw_train_batch_files[0])
    m = re.match(r"^train_batch_\d{4}_(\d{8}_\d{6})\.pkl$", sample_name)
    ACTUAL_BATCH_RUN_ID = m.group(1) if m else RUN_ID
else:
    ACTUAL_BATCH_RUN_ID = RUN_ID

print_line()
print("Metadata RUN_ID:", RUN_ID)
print("Actual batch RUN_ID used:", ACTUAL_BATCH_RUN_ID)
print("Final real train batch files:", len(raw_train_batch_files))
print("Final real test batch files :", len(raw_test_batch_files))
print("Expected train batches from metadata:", metadata.get("num_train_batches"))
print("Expected test batches from metadata :", metadata.get("num_test_batches"))

print("\nSample real train batch files:")
for f in raw_train_batch_files[:5]:
    print(os.path.basename(f))

print("\nSample real test batch files:")
for f in raw_test_batch_files[:5]:
    print(os.path.basename(f))

if len(raw_train_batch_files) == 0 or len(raw_test_batch_files) == 0:
    raise FileNotFoundError("No real train/test batch files found.")

# ============================================================
# 7. LIGHTWEIGHT VALIDATION
# Avoid reading all 2000+ files before training
# ============================================================

train_batch_files = raw_train_batch_files
test_batch_files = raw_test_batch_files

print_line()
print("Using batch files without full validation to avoid Drive disconnect.")
print("Train batch files:", len(train_batch_files))
print("Test batch files :", len(test_batch_files))

REQUIRED_KEYS = {"Xp", "Xa", "Xd", "Xdow", "Xhr", "Xdays", "y"}

sample_train_check = load_pickle(train_batch_files[0])
sample_test_check = load_pickle(test_batch_files[0])

assert isinstance(sample_train_check, dict), "Sample train batch is not a dictionary"
assert isinstance(sample_test_check, dict), "Sample test batch is not a dictionary"
assert REQUIRED_KEYS.issubset(set(sample_train_check.keys())), "Sample train batch missing keys"
assert REQUIRED_KEYS.issubset(set(sample_test_check.keys())), "Sample test batch missing keys"

print("Sample train batch validation passed.")
print("Sample test batch validation passed.")

del sample_train_check, sample_test_check
gc.collect()

# ============================================================
# 8. TRAIN / VALIDATION SPLIT
# ============================================================

set_seed(SEED)

all_train_files = train_batch_files.copy()
random.shuffle(all_train_files)

val_count = max(1, int(len(all_train_files) * VAL_SPLIT))
val_files = all_train_files[:val_count]
train_files = all_train_files[val_count:]

print_line()
print("Train files:", len(train_files))
print("Val files  :", len(val_files))
print("Test files :", len(test_batch_files))

save_json(
    {
        "seed": SEED,
        "val_split": VAL_SPLIT,
        "num_train_files": len(train_files),
        "num_val_files": len(val_files),
        "num_test_files": len(test_batch_files),
        "metadata_run_id": RUN_ID,
        "actual_batch_run_id": ACTUAL_BATCH_RUN_ID
    },
    os.path.join(PHASE4_DIR, "split_info.json")
)

# ============================================================
# 9. INSPECT ONE SAMPLE BATCH
# ============================================================

sample_batch = load_pickle(train_files[0])

print_line()
print("Sample batch file:", os.path.basename(train_files[0]))
print("Sample batch keys:", sample_batch.keys())

for k, v in sample_batch.items():
    arr = np.array(v)
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")

print_line()
print("Xp   min/max:", np.min(sample_batch["Xp"]), np.max(sample_batch["Xp"]))
print("Xa   min/max:", np.min(sample_batch["Xa"]), np.max(sample_batch["Xa"]))
print("Xd   min/max:", np.min(sample_batch["Xd"]), np.max(sample_batch["Xd"]))
print("Xdow min/max:", np.min(sample_batch["Xdow"]), np.max(sample_batch["Xdow"]))
print("Xhr  min/max:", np.min(sample_batch["Xhr"]), np.max(sample_batch["Xhr"]))
print("y    min/max:", np.min(sample_batch["y"]), np.max(sample_batch["y"]))

del sample_batch
gc.collect()

# ============================================================
# 10. SAFE VOCAB SIZES
# ============================================================

PRODUCT_VOCAB_SIZE = 25001
AISLE_VOCAB_SIZE   = 135
DEPT_VOCAB_SIZE    = 22
SAFE_MAX_DOW       = 7
SAFE_MAX_HOUR      = 24

print_line()
print("PRODUCT_VOCAB_SIZE:", PRODUCT_VOCAB_SIZE)
print("AISLE_VOCAB_SIZE  :", AISLE_VOCAB_SIZE)
print("DEPT_VOCAB_SIZE   :", DEPT_VOCAB_SIZE)
print("SAFE_MAX_DOW      :", SAFE_MAX_DOW)
print("SAFE_MAX_HOUR     :", SAFE_MAX_HOUR)

# ============================================================
# 11. DATA LOADER
# ============================================================

def load_batch_tensors(batch_path, device=DEVICE):
    batch = load_pickle(batch_path)

    required_keys = ["Xp", "Xa", "Xd", "Xdow", "Xhr", "Xdays", "y"]
    missing = [k for k in required_keys if k not in batch]

    if missing:
        raise KeyError(f"Missing keys {missing} in file: {batch_path}")

    return {
        "Xp": torch.tensor(batch["Xp"], dtype=torch.long, device=device),
        "Xa": torch.tensor(batch["Xa"], dtype=torch.long, device=device),
        "Xd": torch.tensor(batch["Xd"], dtype=torch.long, device=device),
        "Xdow": torch.tensor(batch["Xdow"], dtype=torch.long, device=device),
        "Xhr": torch.tensor(batch["Xhr"], dtype=torch.long, device=device),
        "Xdays": torch.tensor(batch["Xdays"], dtype=torch.float, device=device),
        "y": torch.tensor(batch["y"], dtype=torch.long, device=device)
    }

def iterate_inner_batches(batch_tensors, inner_batch_size=256):
    n = batch_tensors["y"].shape[0]

    for start in range(0, n, inner_batch_size):
        end = min(start + inner_batch_size, n)

        yield {
            "Xp": batch_tensors["Xp"][start:end],
            "Xa": batch_tensors["Xa"][start:end],
            "Xd": batch_tensors["Xd"][start:end],
            "Xdow": batch_tensors["Xdow"][start:end],
            "Xhr": batch_tensors["Xhr"][start:end],
            "Xdays": batch_tensors["Xdays"][start:end],
            "y": batch_tensors["y"][start:end]
        }

# ============================================================
# 12. METRICS
# ============================================================

def compute_classification_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    return {
        "accuracy": float(acc),
        "precision_weighted": float(prec),
        "recall_weighted": float(rec),
        "f1_weighted": float(f1)
    }

# ============================================================
# 13. MODEL COMPONENTS
# ============================================================

class ReLUGELUActivation(nn.Module):
    def __init__(self, init_alpha=0.5):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(init_alpha, dtype=torch.float32))

    def forward(self, x):
        alpha = torch.clamp(self.alpha, 0.0, 1.0)
        return alpha * F.relu(x) + (1.0 - alpha) * F.gelu(x)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.score = nn.Linear(hidden_dim, 1)

    def forward(self, rnn_outputs, mask=None):
        e = torch.tanh(self.attn(rnn_outputs))
        scores = self.score(e).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        weights = F.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), rnn_outputs).squeeze(1)

        return context, weights

class TimeAwareAttentionRNN(nn.Module):
    def __init__(
        self,
        product_vocab_size,
        aisle_vocab_size,
        dept_vocab_size,
        max_dow,
        max_hour,
        embed_dim_product=64,
        embed_dim_aisle=8,
        embed_dim_dept=4,
        embed_dim_dow=4,
        embed_dim_hour=4,
        time_feature_dim=4,
        rnn_hidden_dim=64,
        dense_dim=64,
        dropout=0.2,
        rnn_type="LSTM",
        activation_type="relu_gelu",
        recency_beta_init=0.10
    ):
        super().__init__()

        self.rnn_type = rnn_type.upper()
        self.activation_type = activation_type.lower()

        self.product_emb = nn.Embedding(product_vocab_size, embed_dim_product, padding_idx=0)
        self.aisle_emb   = nn.Embedding(aisle_vocab_size, embed_dim_aisle, padding_idx=0)
        self.dept_emb    = nn.Embedding(dept_vocab_size, embed_dim_dept, padding_idx=0)
        self.dow_emb     = nn.Embedding(max_dow, embed_dim_dow, padding_idx=0)
        self.hour_emb    = nn.Embedding(max_hour, embed_dim_hour, padding_idx=0)

        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_feature_dim),
            nn.ReLU(),
            nn.Linear(time_feature_dim, time_feature_dim)
        )

        self.recency_beta = nn.Parameter(torch.tensor(recency_beta_init, dtype=torch.float32))

        input_dim = (
            embed_dim_product +
            embed_dim_aisle +
            embed_dim_dept +
            embed_dim_dow +
            embed_dim_hour +
            time_feature_dim +
            1
        )

        if self.rnn_type == "LSTM":
            self.rnn = nn.LSTM(input_size=input_dim, hidden_size=rnn_hidden_dim, batch_first=True)
        elif self.rnn_type == "GRU":
            self.rnn = nn.GRU(input_size=input_dim, hidden_size=rnn_hidden_dim, batch_first=True)
        else:
            raise ValueError("rnn_type must be 'LSTM' or 'GRU'")

        self.attention = AttentionLayer(rnn_hidden_dim)
        self.fc1 = nn.Linear(rnn_hidden_dim, dense_dim)
        self.dropout = nn.Dropout(dropout)

        if self.activation_type == "relu":
            self.activation = nn.ReLU()
        elif self.activation_type == "gelu":
            self.activation = nn.GELU()
        elif self.activation_type == "relu_gelu":
            self.activation = ReLUGELUActivation()
        else:
            raise ValueError("activation_type must be 'relu', 'gelu', or 'relu_gelu'")

        self.fc_out = nn.Linear(dense_dim, product_vocab_size)

    def forward(self, Xp, Xa, Xd, Xdow, Xhr, Xdays):
        p_emb  = self.product_emb(Xp)
        a_emb  = self.aisle_emb(Xa)
        d_emb  = self.dept_emb(Xd)
        dw_emb = self.dow_emb(Xdow)
        hr_emb = self.hour_emb(Xhr)

        xdays_log = torch.log1p(Xdays)
        denom = xdays_log.max().detach() + 1e-8
        xdays_norm = xdays_log / denom

        gap_feature = xdays_norm.unsqueeze(-1)
        time_encoded = self.time_mlp(gap_feature)

        x = torch.cat(
            [p_emb, a_emb, d_emb, dw_emb, hr_emb, time_encoded, gap_feature],
            dim=-1
        )

        beta = torch.clamp(self.recency_beta, min=0.0)
        recency_weight = torch.exp(-beta * xdays_norm).unsqueeze(-1)
        x = x * recency_weight

        mask = (Xp != 0).long()

        rnn_out, _ = self.rnn(x)
        context, attn_weights = self.attention(rnn_out, mask=mask)

        h = self.fc1(context)
        h = self.activation(h)
        h = self.dropout(h)

        logits = self.fc_out(h)

        return logits, attn_weights

def build_model(rnn_type, activation):
    model = TimeAwareAttentionRNN(
        product_vocab_size=PRODUCT_VOCAB_SIZE,
        aisle_vocab_size=AISLE_VOCAB_SIZE,
        dept_vocab_size=DEPT_VOCAB_SIZE,
        max_dow=SAFE_MAX_DOW,
        max_hour=SAFE_MAX_HOUR,
        embed_dim_product=EMBED_DIM_PRODUCT,
        embed_dim_aisle=EMBED_DIM_AISLE,
        embed_dim_dept=EMBED_DIM_DEPT,
        embed_dim_dow=EMBED_DIM_DOW,
        embed_dim_hour=EMBED_DIM_HOUR,
        time_feature_dim=TIME_FEATURE_DIM,
        rnn_hidden_dim=RNN_HIDDEN_DIM,
        dense_dim=DENSE_DIM,
        dropout=DROPOUT,
        rnn_type=rnn_type,
        activation_type=activation,
        recency_beta_init=RECENCY_BETA_INIT
    )

    return model.to(DEVICE)

# ============================================================
# 14. CHECKPOINT HELPERS
# ============================================================

def get_model_dir(model_name):
    model_dir = os.path.join(PHASE4_DIR, model_name)
    ensure_dir(model_dir)
    return model_dir

def get_paths(model_name):
    model_dir = get_model_dir(model_name)

    return {
        "model_dir": model_dir,
        "latest_ckpt": os.path.join(model_dir, "checkpoint_latest.pt"),
        "best_ckpt": os.path.join(model_dir, "checkpoint_best.pt"),
        "history_csv": os.path.join(model_dir, "training_history.csv"),
        "metrics_json": os.path.join(model_dir, "final_metrics.json"),
        "topk_json": os.path.join(model_dir, "topk_metrics.json"),
        "config_json": os.path.join(model_dir, "model_config.json"),
        "done_flag": os.path.join(model_dir, "TRAINING_DONE.flag")
    }

def save_checkpoint(path, epoch, model, optimizer, best_val_f1, history, variant):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_f1": best_val_f1,
        "history": history,
        "variant": variant
    }

    torch.save(checkpoint, path)

def load_checkpoint(path, model, optimizer=None):
    checkpoint = torch.load(path, map_location=DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    epoch = checkpoint.get("epoch", 0)
    best_val_f1 = checkpoint.get("best_val_f1", -1.0)
    history = checkpoint.get("history", [])
    variant = checkpoint.get("variant", None)

    return model, optimizer, epoch, best_val_f1, history, variant

# ============================================================
# 15. TRAIN / EVALUATION FUNCTIONS
# ============================================================

def train_one_epoch(model, optimizer, batch_files, criterion, inner_batch_size=256):
    model.train()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size=inner_batch_size):
            optimizer.zero_grad()

            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            loss = criterion(logits, mini_batch["y"])
            loss.backward()
            optimizer.step()

            batch_size = mini_batch["y"].size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            all_y_true.extend(mini_batch["y"].detach().cpu().numpy().tolist())
            all_y_pred.extend(preds.detach().cpu().numpy().tolist())

            del mini_batch, logits, loss, preds

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % PRINT_EVERY_FILES == 0 or (file_idx + 1) == len(batch_files):
            print(f"Train progress: {file_idx + 1}/{len(batch_files)} files processed")

    epoch_loss = running_loss / total_samples
    metrics = compute_classification_metrics(all_y_true, all_y_pred)

    return epoch_loss, metrics

@torch.no_grad()
def evaluate_model(model, batch_files, criterion, inner_batch_size=256):
    model.eval()

    running_loss = 0.0
    total_samples = 0

    all_y_true = []
    all_y_pred = []

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size=inner_batch_size):
            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            loss = criterion(logits, mini_batch["y"])

            batch_size = mini_batch["y"].size(0)

            running_loss += loss.item() * batch_size
            total_samples += batch_size

            preds = torch.argmax(logits, dim=1)

            all_y_true.extend(mini_batch["y"].detach().cpu().numpy().tolist())
            all_y_pred.extend(preds.detach().cpu().numpy().tolist())

            del mini_batch, logits, loss, preds

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % PRINT_EVERY_FILES == 0 or (file_idx + 1) == len(batch_files):
            print(f"Eval progress: {file_idx + 1}/{len(batch_files)} files processed")

    epoch_loss = running_loss / total_samples
    metrics = compute_classification_metrics(all_y_true, all_y_pred)

    return epoch_loss, metrics

@torch.no_grad()
def evaluate_topk(model, batch_files, k_values=[5, 10], inner_batch_size=64):
    model.eval()

    total = 0
    hits = {k: 0 for k in k_values}

    for file_idx, batch_path in enumerate(batch_files):
        batch_tensors = load_batch_tensors(batch_path, device=DEVICE)

        for mini_batch in iterate_inner_batches(batch_tensors, inner_batch_size=inner_batch_size):
            logits, _ = model(
                mini_batch["Xp"],
                mini_batch["Xa"],
                mini_batch["Xd"],
                mini_batch["Xdow"],
                mini_batch["Xhr"],
                mini_batch["Xdays"]
            )

            y_true = mini_batch["y"].view(-1, 1)

            for k in k_values:
                topk_preds = torch.topk(logits, k=k, dim=1).indices
                hits[k] += (topk_preds == y_true).any(dim=1).sum().item()

            total += y_true.size(0)

            del mini_batch, logits, y_true

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del batch_tensors
        gc.collect()

        if (file_idx + 1) % PRINT_EVERY_FILES == 0 or (file_idx + 1) == len(batch_files):
            print(f"Top-K progress: {file_idx + 1}/{len(batch_files)} test files processed")

    results = {"total_test_samples": total}

    for k in k_values:
        hit_rate = hits[k] / total
        results[f"hit_rate@{k}"] = hit_rate
        results[f"recall@{k}"] = hit_rate
        results[f"precision@{k}"] = hit_rate / k

    return results

# ============================================================
# 16. TRAIN MODEL
# ============================================================

def train_model_variant(variant):
    model_name = variant["name"]
    rnn_type = variant["rnn_type"]
    activation = variant["activation"]

    print_line()
    print(f"STARTING MODEL: {model_name}")
    print_line()

    paths = get_paths(model_name)

    if FORCE_CONTINUE_TRAINING and os.path.exists(paths["done_flag"]):
        os.remove(paths["done_flag"])
        print(f"Removed old done flag for continuation: {model_name}")

    if os.path.exists(paths["done_flag"]):
        print(f"{model_name} already finished earlier. Skipping training.")
        return

    model = build_model(rnn_type, activation)

    print(f"Trainable params: {count_parameters(model):,}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    criterion = nn.CrossEntropyLoss()

    start_epoch = 1
    best_val_f1 = -1.0
    history = []

    save_json(
        {
            "model_name": model_name,
            "rnn_type": rnn_type,
            "activation": activation,
            "num_epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "dropout": DROPOUT,
            "seed": SEED,
            "product_vocab_size": PRODUCT_VOCAB_SIZE,
            "aisle_vocab_size": AISLE_VOCAB_SIZE,
            "dept_vocab_size": DEPT_VOCAB_SIZE,
            "safe_max_dow": SAFE_MAX_DOW,
            "safe_max_hour": SAFE_MAX_HOUR,
            "inner_batch_size": INNER_BATCH_SIZE
        },
        paths["config_json"]
    )

    if os.path.exists(paths["latest_ckpt"]):
        print(f"Resuming from checkpoint: {paths['latest_ckpt']}")

        model, optimizer, last_epoch, best_val_f1, history, _ = load_checkpoint(
            paths["latest_ckpt"],
            model,
            optimizer
        )

        start_epoch = last_epoch + 1

        for param_group in optimizer.param_groups:
            param_group["lr"] = LEARNING_RATE

        print(f"Resumed from epoch {last_epoch}. New LR set to {LEARNING_RATE}")

        if start_epoch > NUM_EPOCHS:
            print(f"{model_name} already reached target epochs. Marking done.")

            with open(paths["done_flag"], "w") as f:
                f.write("done")

            return

    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        print_line()
        print(f"{model_name} | Epoch {epoch}/{NUM_EPOCHS}")

        start_time = time.time()

        train_loss, train_metrics = train_one_epoch(
            model,
            optimizer,
            train_files,
            criterion,
            inner_batch_size=INNER_BATCH_SIZE
        )

        val_loss, val_metrics = evaluate_model(
            model,
            val_files,
            criterion,
            inner_batch_size=INNER_BATCH_SIZE
        )

        elapsed = time.time() - start_time

        epoch_record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_accuracy": train_metrics["accuracy"],
            "train_precision_weighted": train_metrics["precision_weighted"],
            "train_recall_weighted": train_metrics["recall_weighted"],
            "train_f1_weighted": train_metrics["f1_weighted"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision_weighted": val_metrics["precision_weighted"],
            "val_recall_weighted": val_metrics["recall_weighted"],
            "val_f1_weighted": val_metrics["f1_weighted"],
            "epoch_time_sec": elapsed
        }

        history.append(epoch_record)

        print(f"Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        print(f"Train Acc : {train_metrics['accuracy']:.6f} | Val Acc : {val_metrics['accuracy']:.6f}")
        print(f"Train F1  : {train_metrics['f1_weighted']:.6f} | Val F1  : {val_metrics['f1_weighted']:.6f}")
        print(f"Epoch time: {elapsed:.2f} sec")

        save_checkpoint(
            paths["latest_ckpt"],
            epoch,
            model,
            optimizer,
            best_val_f1,
            history,
            variant
        )

        if val_metrics["f1_weighted"] > best_val_f1:
            best_val_f1 = val_metrics["f1_weighted"]

            save_checkpoint(
                paths["best_ckpt"],
                epoch,
                model,
                optimizer,
                best_val_f1,
                history,
                variant
            )

            print("Saved new BEST checkpoint.")

        pd.DataFrame(history).to_csv(paths["history_csv"], index=False)

    with open(paths["done_flag"], "w") as f:
        f.write("done")

    print_line()
    print(f"TRAINING COMPLETE: {model_name}")

# ============================================================
# 17. FINAL TEST EVALUATION
# ============================================================

def final_test_evaluation(variant):
    model_name = variant["name"]
    paths = get_paths(model_name)

    if not os.path.exists(paths["best_ckpt"]):
        print(f"No best checkpoint found for {model_name}. Skipping test evaluation.")
        return None

    model = build_model(
        variant["rnn_type"],
        variant["activation"]
    )

    criterion = nn.CrossEntropyLoss()

    model, _, best_epoch, best_val_f1, history, _ = load_checkpoint(
        paths["best_ckpt"],
        model,
        optimizer=None
    )

    test_loss, test_metrics = evaluate_model(
        model,
        test_batch_files,
        criterion,
        inner_batch_size=INNER_BATCH_SIZE
    )

    topk_metrics = evaluate_topk(
        model,
        test_batch_files,
        k_values=[5, 10],
        inner_batch_size=TOPK_INNER_BATCH_SIZE
    )

    result = {
        "model_name": model_name,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1,
        "test_loss": test_loss,
        "test_accuracy": test_metrics["accuracy"],
        "test_precision_weighted": test_metrics["precision_weighted"],
        "test_recall_weighted": test_metrics["recall_weighted"],
        "test_f1_weighted": test_metrics["f1_weighted"],
        **topk_metrics
    }

    save_json(result, paths["metrics_json"])

    print_line()
    print(f"FINAL TEST RESULTS: {model_name}")

    for k, v in result.items():
        print(f"{k}: {v}")

    return result

# ============================================================
# 18. RUN TRAINING
# ============================================================

for variant in MODEL_VARIANTS:
    train_model_variant(variant)

# ============================================================
# 19. RUN FINAL EVALUATION
# ============================================================

all_results = []

for variant in MODEL_VARIANTS:
    result = final_test_evaluation(variant)

    if result is not None:
        all_results.append(result)

results_df = pd.DataFrame(all_results)

comparison_csv = os.path.join(PHASE4_DIR, "phase4_lstm_relu_gelu_comparison_with_topk.csv")
comparison_json = os.path.join(PHASE4_DIR, "phase4_lstm_relu_gelu_comparison_with_topk.json")

results_df.to_csv(comparison_csv, index=False)
save_json(all_results, comparison_json)

print_line()
print("PHASE 4 LSTM + RELU-GELU COMPLETE")
print_line()

print("Saved final results to:")
print(comparison_csv)
print(comparison_json)

if len(results_df) > 0:
    display(results_df.sort_values(by="test_f1_weighted", ascending=False))
else:
    print("No model results found.")

Mounted at /content/drive
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Using device: cuda
Using Phase 3 directly from Drive:
/content/drive/MyDrive/Instacart/phase3_outputs_final
Loaded metadata from: /content/drive/MyDrive/Instacart/phase3_outputs_final/phase3_metadata.pkl
RUN_ID: 20260429_204444
MAX_LEN: 50
CHUNK_SIZE: 100
num_train_users: 164331
num_test_users: 41083
num_train_batches: 1644
num_test_batches: 411
product_vocab_size: 25001
aisle_vocab_size: 134
dept_vocab_size: 21
sample_train_batch_file: train_batch_0001_20260422_185902.pkl
Metadata RUN_ID: 20260429_204444
Discovering real Phase 3 batch files...
Real train batches for metadata RUN_ID: 0
Real test batches for metadata RUN_ID : 0

No real batch files found for metadata RUN_ID.
Searching all real batch files from any RUN_ID...
Metadata RUN_ID: 20260429_204444
Actual batch RUN_ID used: 20260422_185902
Final real train batch files: 1644
Final real test batch files : 411
Expected train batches from meta

,model_name,best_epoch,best_val_f1,test_loss,test_accuracy,test_precision_weighted,test_recall_weighted,test_f1_weighted,total_test_samples,hit_rate@5,recall@5,precision@5,hit_rate@10,recall@10,precision@10
0,Light_TimeAwareAttentionLSTM_ReLU_GELU,5,0.011758,7.38819,0.036138,0.018288,0.036138,0.01184,6348182,0.096233,0.096233,0.019247,0.13956,0.13956,0.013956
